# 🧠 Day 4 — Building & Training a Network in Keras

**Phase 3 — Fraud Detection Project**

Today we move from the neural-network concepts of Day 3 to an actual implementation with **TensorFlow / Keras**.

## 🎯 Learning Objectives
- Build a neural network with the Keras Sequential API.
- Compile, train, and evaluate the network.
- Read the training history and diagnose the fit.
- Apply Batch Normalization and Dropout.
- Compare the new model with the previous network and the Day 1 baseline.


## 📚 Key Workflow

**Build → Compile → Fit → Evaluate**

This is a binary classification problem, so the output layer uses **Sigmoid** and the loss function is **Binary Cross-Entropy**.


In [ ]:
# 📦 Imports + reproducibility
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


## 📂 1. Load the Project Dataset

The Phase 3 project uses the real **Credit Card Fraud Detection** dataset.

- `Class = 0` → normal transaction
- `Class = 1` → fraudulent transaction


In [ ]:
# 📥 Load the dataset: local file first, then Google Drive as a fallback
# (the full dataset is ~98MB, too large to commit to GitHub directly,
# so it's hosted on Google Drive and downloaded here at runtime)
from pathlib import Path

DATA_PATH = Path("creditcard.csv")

if not DATA_PATH.exists():
    try:
        import gdown
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown

    GDRIVE_FILE_ID = "1UZ3hAYkcXulu8MSdZQiSzpgb1qu4bXNh"
    gdown.download(
        f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}",
        str(DATA_PATH),
        quiet=False,
    )

if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/creditcard.csv")

df = pd.read_csv(DATA_PATH)

print(f"Using dataset: {DATA_PATH}")
print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["Class"].value_counts().sort_index())

assert "Class" in df.columns
assert df["Class"].nunique() == 2


## 🔎 2. Data Quality & Leakage Checks

Before training:

- Check missing values.
- Check infinite values.
- Separate the target from the features.
- Make sure `Class` is not used as an input feature.


In [ ]:
numeric = df.select_dtypes(include=np.number)

missing = int(df.isna().sum().sum())
infinite = int(np.isinf(numeric).sum().sum())
duplicates = int(df.duplicated().sum())

print("Missing values:", missing)
print("Infinite values:", infinite)
print("Duplicate rows:", duplicates)

assert missing == 0
assert infinite == 0

X = df.drop(columns=["Class"])
y = df["Class"].astype(np.int32)

assert "Class" not in X.columns
print("Input features:", X.shape[1])


## ✂️ 3. Train / Validation / Test Split

We use:

- **70% training**
- **15% validation**
- **15% test**

Stratification preserves the class ratio across the splits.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nFraud rate:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())


## 📏 4. Feature Scaling Without Leakage

The scaler is fitted **only on the training set**. Validation and test data are transformed using the training-set statistics.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()
assert np.isfinite(X_test_scaled).all()

print("Scaled shapes:")
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)


## 📊 5. Day 1 Baseline

We reproduce a Logistic Regression baseline on the same train/test split.

Because fraud is highly imbalanced, we report **Accuracy, ROC-AUC, and PR-AUC** rather than relying on accuracy alone.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

baseline.fit(X_train, y_train)

baseline_proba = baseline.predict_proba(X_test)[:, 1]
baseline_pred = (baseline_proba >= 0.5).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_roc_auc = roc_auc_score(y_test, baseline_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)

print("Day 1 Baseline")
print("Accuracy:", baseline_accuracy)
print("ROC-AUC:", baseline_roc_auc)
print("PR-AUC:", baseline_pr_auc)


# 🖥️ Hands-On Lab — Training a Neural Network

### Step 1
Build a Keras Sequential network appropriate for the Fraud Detection task.

### Step 2
Compile with Adam and the correct loss, then train with validation data for at least 30 epochs.

### Step 3
Plot training vs. validation loss and accuracy and diagnose the fit.

### Step 4
Add Dropout and/or Batch Normalization and compare the new curves.

### Step 5
Evaluate on the test set and compare the score to the Day 1 baseline.


## 🏗️ Step 1 — Build the First Sequential Network

For binary classification:

- Hidden layers → ReLU
- Output layer → one Sigmoid neuron


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="fraud_detection_nn")

model.summary()


## ⚙️ Step 2 — Compile and Train

Following the lesson:

- Optimizer: **Adam**
- Loss: **Binary Cross-Entropy**
- Metric: **Accuracy**
- Epochs: **30**
- Batch size: **32**


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

assert len(history.history["loss"]) == 30
print("Training completed.")


## 📈 Step 3 — Read the Training History


In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())


In [ ]:
# Training vs validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Original Network — Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# Training vs validation accuracy
plt.figure(figsize=(10, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Original Network — Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### 🔍 Fit Diagnosis

Compare training and validation behavior:

- Both losses remain high → possible underfitting.
- Training loss falls while validation loss rises → possible overfitting.
- Both improve together → evidence that the model is learning useful patterns.

The curves are the evidence used for today's diagnosis.


## 🛡️ Step 4 — Add Batch Normalization + Dropout

The second model adds:

- `BatchNormalization()` for more stable optimization.
- `Dropout(0.30)` as regularization to reduce overfitting.


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)

regularized_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="fraud_detection_regularized_nn")

regularized_model.summary()


In [ ]:
regularized_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

regularized_history = regularized_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

assert len(regularized_history.history["loss"]) == 30
print("Regularized training completed.")


### 📊 Compare Validation Loss


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["val_loss"], label="Original — Validation Loss")
plt.plot(
    regularized_history.history["val_loss"],
    label="BatchNorm + Dropout — Validation Loss"
)
plt.title("Validation Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### 📊 Compare Validation Accuracy


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["val_accuracy"], label="Original — Validation Accuracy")
plt.plot(
    regularized_history.history["val_accuracy"],
    label="BatchNorm + Dropout — Validation Accuracy"
)
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Original NN", "BatchNorm + Dropout NN"],
    "Best Validation Loss": [
        min(history.history["val_loss"]),
        min(regularized_history.history["val_loss"])
    ],
    "Best Validation Accuracy": [
        max(history.history["val_accuracy"]),
        max(regularized_history.history["val_accuracy"])
    ]
})

display(comparison)


## 🧪 Step 5 — Evaluate on the Test Set

The test set has not been used for training.

We evaluate both neural networks and compare them with the Day 1 baseline.


In [ ]:
# Original neural network
original_loss, original_accuracy = model.evaluate(
    X_test_scaled, y_test, verbose=0
)
original_proba = model.predict(X_test_scaled, verbose=0).ravel()

original_roc_auc = roc_auc_score(y_test, original_proba)
original_pr_auc = average_precision_score(y_test, original_proba)

# Regularized neural network
regularized_loss, regularized_accuracy = regularized_model.evaluate(
    X_test_scaled, y_test, verbose=0
)
regularized_proba = regularized_model.predict(
    X_test_scaled, verbose=0
).ravel()

regularized_roc_auc = roc_auc_score(y_test, regularized_proba)
regularized_pr_auc = average_precision_score(y_test, regularized_proba)

print("Original NN")
print("Test loss:", original_loss)
print("Test accuracy:", original_accuracy)
print("ROC-AUC:", original_roc_auc)
print("PR-AUC:", original_pr_auc)

print("\nBatchNorm + Dropout NN")
print("Test loss:", regularized_loss)
print("Test accuracy:", regularized_accuracy)
print("ROC-AUC:", regularized_roc_auc)
print("PR-AUC:", regularized_pr_auc)


In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "Day 1 Logistic Regression",
        "Day 4 Original NN",
        "Day 4 BatchNorm + Dropout NN"
    ],
    "Accuracy": [
        baseline_accuracy,
        original_accuracy,
        regularized_accuracy
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        original_roc_auc,
        regularized_roc_auc
    ],
    "PR-AUC": [
        baseline_pr_auc,
        original_pr_auc,
        regularized_pr_auc
    ],
    "Test Loss": [
        np.nan,
        original_loss,
        regularized_loss
    ]
})

display(final_comparison)


## 🔬 Objective → Evidence Mapping

| Objective | Evidence |
|---|---|
| Build Keras Sequential model | Sequential model with Dense layers |
| Correct output activation | Sigmoid output for binary classification |
| Compile and train | Adam + Binary Cross-Entropy + 30 epochs |
| Read training history | History DataFrame + loss/accuracy curves |
| Diagnose the fit | Training/validation comparison |
| Apply Batch Normalization | Regularized model |
| Apply Dropout | `Dropout(0.30)` |
| Compare models | Validation curves + test metrics |
| Compare with Day 1 | Final comparison table |


## 🧪 Data-Quality, Leakage & Reproducibility Checks


In [ ]:
assert "Class" not in X.columns
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0

for arr in [X_train_scaled, X_val_scaled, X_test_scaled]:
    assert np.isfinite(arr).all()

print("All final sanity checks: PASSED")


## 💭 Reflection

Today the neural-network workflow became concrete: Keras handles the forward pass, loss calculation, backpropagation, and optimizer updates behind the high-level API.

The training history is especially useful because it gives evidence about how the model behaves during training rather than only showing a final score.

Batch Normalization and Dropout also connect today's work directly to the regularization concepts covered earlier.


## 🚀 Beyond the Requirement

Accuracy alone can be misleading for fraud detection because fraudulent transactions are rare.

For that reason, this notebook reports **ROC-AUC and PR-AUC** in addition to accuracy and test loss. This gives a stronger basis for comparing the neural network with the Day 1 baseline.


## 📝 Day 4 Conclusion

The first Keras neural network for the Fraud Detection project has been built, trained, evaluated, and compared with a regularized version and the Day 1 baseline.

The final model choice should be based on the actual validation curves and test-set evidence rather than model complexity alone.


## ✅ Day 4 Completion Checklist

- [x] TensorFlow / Keras
- [x] Real project dataset
- [x] Data-quality checks
- [x] Leakage-aware splitting and scaling
- [x] Sequential API
- [x] Dense layers
- [x] Adam optimizer
- [x] Binary Cross-Entropy
- [x] At least 30 epochs
- [x] Validation data
- [x] Training history
- [x] Loss curves
- [x] Accuracy curves
- [x] Batch Normalization
- [x] Dropout
- [x] Model comparison
- [x] Test-set evaluation
- [x] Day 1 baseline comparison
- [x] Sanity checks
